# YOLOv8s Mob 检测器训练

本Notebook用于训练Florr.io游戏中的怪物检测模型。

## 平台信息
- GPU: P100/T4 (Kaggle) 或 T4 (Colab)
- 模型: YOLOv8s
- 数据集: 81类怪物

## 1. 环境设置

In [ ]:
# 检查GPU
import torch
print(f"PyTorch版本: {torch.__version__}")
print(f"CUDA可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU型号: {torch.cuda.get_device_name(0)}")
    print(f"显存: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

In [ ]:
# 安装依赖
!pip install -q ultralytics
print("✓ Ultralytics安装完成")

## 2. 数据集准备

### Kaggle用户:
1. 在右侧点击 'Add Data'
2. 搜索并添加你的数据集
3. 修改下方 `data_yaml` 路径

### Colab用户:
1. 上传数据集到Google Drive
2. 或使用 `from google.colab import files` 上传
3. 修改下方 `data_yaml` 路径

In [ ]:
# 设置数据集路径

# Kaggle路径（修改为你的数据集名称）
data_yaml = '/kaggle/input/florr-mob-dataset/data.yaml'

# Colab路径（如果使用Colab，取消注释下面这行）
# data_yaml = '/content/drive/MyDrive/florr_dataset/data.yaml'

# 检查数据集是否存在
import os
if os.path.exists(data_yaml):
    print(f"✓ 数据集配置文件存在: {data_yaml}")
    
    # 显示配置内容
    with open(data_yaml, 'r') as f:
        print("\n数据集配置:")
        print(f.read())
else:
    print(f"✗ 数据集配置文件不存在: {data_yaml}")
    print("请先添加数据集并修改路径")

## 3. 开始训练

In [ ]:
from ultralytics import YOLO
import os

# 加载YOLOv8s预训练模型
print("加载YOLOv8s模型...")
model = YOLO('yolov8s.pt')
print("✓ 模型加载完成")

# 训练参数
train_config = {
    'data': data_yaml,
    'epochs': 100,
    'imgsz': 640,
    'batch': 16,  # 如果显存不足，改为8或4
    'name': 'mob_detector_yolov8s',
    'project': './runs',
    'device': 'cuda',
    'patience': 20,  # 早停耐心值
    'save': True,
    'plots': True,
    'optimizer': 'AdamW',
    'lr0': 0.001,  # 初始学习率
    'lrf': 0.01,   # 最终学习率系数
    'weight_decay': 0.0005,
    'warmup_epochs': 3,
    'box': 7.5,    # 边框损失权重
    'cls': 0.5,    # 分类损失权重
    'dfl': 1.5,    # 分布焦点损失权重
    'hsv_h': 0.015,  # HSV增强
    'hsv_s': 0.7,
    'hsv_v': 0.4,
    'translate': 0.1,  # 平移增强
    'scale': 0.5,      # 缩放增强
    'fliplr': 0.5,     # 水平翻转
    'mosaic': 1.0,     # Mosaic增强
    'workers': 4,
    'amp': True,  # 混合精度训练
}

print("\n训练配置:")
for k, v in train_config.items():
    print(f"  {k}: {v}")

print("\n开始训练...")
print("=" * 60)

In [ ]:
# 开始训练
results = model.train(**train_config)

print("\n" + "=" * 60)
print("✓ 训练完成!")

## 4. 模型验证

In [ ]:
# 验证模型
print("验证模型...")
metrics = model.val()

print(f"\n验证结果:")
print(f"  mAP50: {metrics.box.map50:.4f}")
print(f"  mAP50-95: {metrics.box.map:.4f}")
print(f"  Precision: {metrics.box.mp:.4f}")
print(f"  Recall: {metrics.box.mr:.4f}")

## 5. 保存模型

In [ ]:
import shutil
from pathlib import Path

# 模型路径
best_model = Path('./runs/mob_detector_yolov8s/weights/best.pt')
last_model = Path('./runs/mob_detector_yolov8s/weights/last.pt')

print(f"最佳模型: {best_model}")
print(f"最后模型: {last_model}")

# Kaggle: 保存到output目录
output_dir = Path('/kaggle/working/')
if output_dir.exists():
    print("\n保存到Kaggle output...")
    shutil.copy(best_model, output_dir / 'best.pt')
    shutil.copy(last_model, output_dir / 'last.pt')
    
    # 打包整个训练结果
    shutil.make_archive('/kaggle/working/mob_detector_results', 'zip', 
                        './runs/mob_detector_yolov8s')
    print("✓ 模型已保存到 /kaggle/working/")

# Colab: 保存到Google Drive
drive_dir = Path('/content/drive/MyDrive/models/')
if drive_dir.parent.exists():
    print("\n保存到Google Drive...")
    drive_dir.mkdir(exist_ok=True)
    shutil.copy(best_model, drive_dir / 'mob_detector_best.pt')
    print(f"✓ 模型已保存到 {drive_dir}")

## 6. 测试模型（可选）

In [ ]:
# 测试单张图片
test_image = '/kaggle/input/florr-mob-dataset/val/images/example.jpg'  # 修改为实际路径

if os.path.exists(test_image):
    print(f"测试图片: {test_image}")
    results = model.predict(test_image, save=True)
    
    # 显示结果
    from IPython.display import Image, display
    display(Image(filename=results[0].save_dir / results[0].path.name))
else:
    print(f"测试图片不存在: {test_image}")

## 7. 下载模型

### Kaggle用户:
- 在右侧 'Output' 区域找到并下载模型文件

### Colab用户:
- 模型已保存到Google Drive
- 或使用以下代码下载:
```python
from google.colab import files
files.download('runs/mob_detector_yolov8s/weights/best.pt')
```